# Assignment 5

## data import
I am importing the data via sklean.datasets
The raw data from the website was in a format that would have been a little more 
complicated to insert as raw data in column format so I looked up the same datasets
for sklearn and I found that sklearn.datasets contains this data in a usable format

In [ ]:
from sklearn.datasets import fetch_openml

data = fetch_openml(data_id=37, as_frame=True)

X,y = data.data, data.target

## Checking out the Data
The head shows some rows of numerical columns that contain values of 0
This makes no sense for certain parts of the data like insulin and bmi where it's impossible for them to be 0. If they were, the person would be dead

In [ ]:
X.info()

In [ ]:
X.head()

In [ ]:
X.describe()

## Removing zero's in columns that make no sense
Here, we are going through the columns that would be impossible for it to be 0 and changing them to np.nan so the imputer knows to impute on them

In [ ]:
import numpy as np

columns = ['skin', 'insu', 'plas', 'mass', 'pres']

for col in columns:
    X[col] = X[col].replace(0, np.nan)

X.describe()

X.isna().sum()

## Checking the target df for missing values

all good here. just checking for values that wouldn't make sense
Also turning y value data into numeric values for the tests later on

In [ ]:
y.describe()
y = (y == 'tested_positive').astype(int)

## Pipeline definition

X data is nothing but numerical values so I don't have to split categorical and numerical data
But y is only categorical so I will still need a cat pipeline
Using median imputation because, as seen in the X.skew() cell, some of the data is heavily skewed, so mean imputation isn't good here
Otherwise, we are just using default values.


In [ ]:
X.skew()

In [ ]:
numCols = list(X.columns)
catCols = []

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer

numPipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
catPipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('cat_encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessPipe = ColumnTransformer([
    ('num', numPipe, numCols),
    ('cat', catPipe, catCols)
])

## Defining Voting Classifier

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone


votingClfPipe = Pipeline([
    ('preprocess', clone(preprocessPipe)),
    ('VotingClf', VotingClassifier(
            estimators=[
                ('lr', LogisticRegression(random_state=42, max_iter=1000)),
                ('SVM', SVC(kernel='rbf', probability=True, random_state=42)),
                ('tree', DecisionTreeClassifier(max_depth=5, random_state=42))
            ]
        )
    )
])

Xtrain, Xtest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

votingClfPipe.fit_transform(Xtrain, yTrain)
votingClfPipe.predict(Xtest)

Testing each individual model to see how good they are compared to the voting method
I have to send the Xtest through the pipeline to get rid of nan values first
I also had to get the classifier from the pipeline before doing any testing

In [ ]:
vc = votingClfPipe.named_steps['VotingClf']
XtestReady = votingClfPipe.named_steps['preprocess'].transform(Xtest)
for name, clf in vc.named_estimators_.items():
    print(name, "=", clf.score(XtestReady, yTest))

The voting classifier sits in the middle of all the models after scoring it's performance

In [ ]:
votingClfPipe[-1].voting = "hard"
votingClfPipe.predict(Xtest)
hard = votingClfPipe.score(Xtest,yTest)

same results after using soft voting
I also wanted to check how much they differ and we can see they don't differ at all. This means all the models predicted the same for both testers. 
It also means the models made the same mistakes on the dataset

In [ ]:
votingClfPipe[-1].voting = "soft"
votingClfPipe.predict(Xtest)
soft = votingClfPipe.score(Xtest, yTest)

print("rows that differ:", np.sum(hard != soft))